# 04: Error Analysis and Failure Buckets

Scores tell you whether a system is winning or losing. Error analysis tells you why. This notebook focuses on turning raw failures into categories, severity buckets, and concrete roadmap decisions.

## Learning goals

- build a practical failure taxonomy
- assign severity levels instead of treating every miss equally
- summarize recurring error patterns across a run
- compare versions by failure type, not only by average score
- translate failure analysis into next engineering actions

In [ ]:
# --- Google Colab Setup ---
!pip install -q "transformers>=4.51.0" accelerate bitsandbytes torch

import csv
import json
import math
import random
import statistics
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True, top_k=20):
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=top_k,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated_ids = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## Why failure buckets matter

A benchmark score can hide two very different realities:

- many harmless formatting mistakes
- a few severe security or policy failures

Both might produce the same average. That is why mature eval loops separate:

- what failed
- how often it failed
- how severe the failure was
- what intervention is most likely to fix it

## We will work with a mock experiment

The experiment compares two prompt versions across routing, extraction, and answer-generation tasks. The data is small on purpose so the analysis stays transparent.

In [ ]:

def print_table(rows, headers=None):
    rows = list(rows)
    if not rows:
        print("No rows")
        return

    if headers is None:
        if isinstance(rows[0], dict):
            headers = list(rows[0].keys())
        else:
            headers = [f"col_{index}" for index in range(len(rows[0]))]

    normalized = []
    for row in rows:
        if isinstance(row, dict):
            normalized.append([str(row.get(header, "")) for header in headers])
        else:
            normalized.append([str(value) for value in row])

    widths = []
    for index, header in enumerate(headers):
        content_width = max(len(values[index]) for values in normalized)
        widths.append(max(len(str(header)), content_width))

    header_line = " | ".join(str(header).ljust(widths[index]) for index, header in enumerate(headers))
    divider = "-+-".join("-" * width for width in widths)

    print(header_line)
    print(divider)
    for values in normalized:
        print(" | ".join(values[index].ljust(widths[index]) for index in range(len(headers))))


In [ ]:

mock_run = [
    {
        "id": "r1",
        "scenario": "former_vendor_access",
        "version": "baseline",
        "task_type": "routing",
        "gold": "security",
        "prediction": "account_access",
        "format_valid": True,
        "unsupported_claim": False,
        "missing_critical": False,
        "policy_violation": False,
        "notes": "Sent former vendor access case to password reset workflow."
    },
    {
        "id": "r2",
        "scenario": "invoice_due_date_extraction",
        "version": "baseline",
        "task_type": "extraction",
        "gold": ["invoice_number", "amount", "due_date"],
        "prediction": ["invoice_number", "amount"],
        "format_valid": True,
        "unsupported_claim": False,
        "missing_critical": True,
        "policy_violation": False,
        "notes": "Dropped due_date when it appeared in the footer."
    },
    {
        "id": "r3",
        "scenario": "refund_policy_answer",
        "version": "baseline",
        "task_type": "answering",
        "gold": "Refunds require a receipt and must be requested within 30 days.",
        "prediction": "Enterprise customers can request a refund at any time without a receipt.",
        "format_valid": True,
        "unsupported_claim": True,
        "missing_critical": True,
        "policy_violation": False,
        "notes": "Invented a generous refund policy not supported by the reference."
    },
    {
        "id": "r4",
        "scenario": "dashboard_bug_routing",
        "version": "baseline",
        "task_type": "routing",
        "gold": "technical_issue",
        "prediction": "This sounds like a bug for engineering to inspect.",
        "format_valid": False,
        "unsupported_claim": False,
        "missing_critical": False,
        "policy_violation": False,
        "notes": "Returned prose instead of one routing label."
    },
    {
        "id": "r5",
        "scenario": "password_reset_answer",
        "version": "baseline",
        "task_type": "answering",
        "gold": "Verify identity, trigger a reset email, and confirm delivery troubleshooting if needed.",
        "prediction": "Tell the user to click forgot password and wait.",
        "format_valid": True,
        "unsupported_claim": False,
        "missing_critical": True,
        "policy_violation": False,
        "notes": "Skipped identity verification and delivery troubleshooting steps."
    },
    {
        "id": "r6",
        "scenario": "contract_field_extraction",
        "version": "baseline",
        "task_type": "extraction",
        "gold": ["vendor_name", "renewal_date", "termination_notice_days"],
        "prediction": ["vendor_name", "renewal_date", "termination_notice_days"],
        "format_valid": True,
        "unsupported_claim": False,
        "missing_critical": False,
        "policy_violation": False,
        "notes": "All required fields captured."
    },
    {
        "id": "r7",
        "scenario": "former_vendor_access",
        "version": "prompt_v2",
        "task_type": "routing",
        "gold": "security",
        "prediction": "security",
        "format_valid": True,
        "unsupported_claim": False,
        "missing_critical": False,
        "policy_violation": False,
        "notes": "Correctly escalated unauthorized access."
    },
    {
        "id": "r8",
        "scenario": "invoice_due_date_extraction",
        "version": "prompt_v2",
        "task_type": "extraction",
        "gold": ["invoice_number", "amount", "due_date"],
        "prediction": ["invoice_number", "amount", "due_date"],
        "format_valid": True,
        "unsupported_claim": False,
        "missing_critical": False,
        "policy_violation": False,
        "notes": "Footer instruction fixed the missing field."
    },
    {
        "id": "r9",
        "scenario": "refund_policy_answer",
        "version": "prompt_v2",
        "task_type": "answering",
        "gold": "Refunds require a receipt and must be requested within 30 days.",
        "prediction": "Refunds require a receipt and must be requested within 30 days.",
        "format_valid": True,
        "unsupported_claim": False,
        "missing_critical": False,
        "policy_violation": False,
        "notes": "Reference-grounded answer with no invented detail."
    },
    {
        "id": "r10",
        "scenario": "dashboard_bug_routing",
        "version": "prompt_v2",
        "task_type": "routing",
        "gold": "technical_issue",
        "prediction": "technical_issue",
        "format_valid": True,
        "unsupported_claim": False,
        "missing_critical": False,
        "policy_violation": False,
        "notes": "Formatting guardrail forced a clean label."
    },
    {
        "id": "r11",
        "scenario": "password_reset_answer",
        "version": "prompt_v2",
        "task_type": "answering",
        "gold": "Verify identity, trigger a reset email, and confirm delivery troubleshooting if needed.",
        "prediction": "Reset the password immediately for the user so they can regain access faster.",
        "format_valid": True,
        "unsupported_claim": False,
        "missing_critical": False,
        "policy_violation": True,
        "notes": "Suggested bypassing identity verification for an account recovery flow."
    },
    {
        "id": "r12",
        "scenario": "contract_field_extraction",
        "version": "prompt_v2",
        "task_type": "extraction",
        "gold": ["vendor_name", "renewal_date", "termination_notice_days"],
        "prediction": ["vendor_name", "renewal_date"],
        "format_valid": True,
        "unsupported_claim": False,
        "missing_critical": True,
        "policy_violation": False,
        "notes": "Still misses termination_notice_days in long contracts."
    },
]

print("Rows in mock experiment:", len(mock_run))
print_table(mock_run[:6], headers=["id", "scenario", "version", "task_type", "gold", "prediction"])


## Define the failure taxonomy before reviewing examples

A taxonomy does not need to be perfect on day one. It just needs to be stable enough that two reviewers can usually place the same failure in the same bucket.

In [ ]:

severity_order = {
    "low": 1,
    "medium": 2,
    "high": 3,
    "critical": 4,
}

bucket_metadata = {
    "pass": {"severity": "low", "meaning": "Meets the task requirement"},
    "wrong_label": {"severity": "medium", "meaning": "Predicted the wrong route or class"},
    "missing_required_field": {"severity": "medium", "meaning": "Dropped a required extracted field"},
    "partial_answer": {"severity": "medium", "meaning": "Answer is incomplete or omits a key step"},
    "hallucinated_detail": {"severity": "high", "meaning": "Introduced unsupported content"},
    "formatting_error": {"severity": "low", "meaning": "Output format violates the contract"},
    "policy_violation": {"severity": "critical", "meaning": "Unsafe or disallowed recommendation"},
    "other_failure": {"severity": "medium", "meaning": "Failure not yet covered by current rules"},
}

print_table(
    [
        {"bucket": bucket, "severity": details["severity"], "meaning": details["meaning"]}
        for bucket, details in bucket_metadata.items()
    ],
    headers=["bucket", "severity", "meaning"],
)


## Build simple bucketing helpers

These helpers are deliberately transparent. You should be able to read them line by line and understand why a failure ended up in a given category.

In [ ]:

def is_pass(record):
    if record["task_type"] == "routing":
        return record["format_valid"] and record["prediction"] == record["gold"]

    if record["task_type"] == "extraction":
        return record["format_valid"] and set(record["prediction"]) == set(record["gold"])

    if record["task_type"] == "answering":
        return (
            record["format_valid"]
            and not record["unsupported_claim"]
            and not record["missing_critical"]
            and not record["policy_violation"]
        )

    return False

def bucket_error(record):
    if is_pass(record):
        return "pass"
    if record["policy_violation"]:
        return "policy_violation"
    if not record["format_valid"]:
        return "formatting_error"
    if record["task_type"] == "routing" and record["prediction"] != record["gold"]:
        return "wrong_label"
    if record["unsupported_claim"]:
        return "hallucinated_detail"
    if record["task_type"] == "extraction" and record["missing_critical"]:
        return "missing_required_field"
    if record["task_type"] == "answering" and record["missing_critical"]:
        return "partial_answer"
    return "other_failure"

def annotate_record(record):
    annotated = dict(record)
    annotated["bucket"] = bucket_error(record)
    annotated["severity"] = bucket_metadata[annotated["bucket"]]["severity"]
    annotated["passed"] = int(annotated["bucket"] == "pass")
    annotated["severity_score"] = severity_order[annotated["severity"]]
    return annotated

annotated_run = [annotate_record(record) for record in mock_run]


## Apply the taxonomy to the mock experiment

Once the run is annotated, we can slice it in several useful ways: by version, by task type, by bucket, and by severity.

In [ ]:

print_table(
    annotated_run,
    headers=["id", "version", "task_type", "scenario", "bucket", "severity", "passed"],
)


## Count recurring error patterns

The first useful summary is usually just counts. Which failure families dominate the run? Which ones disappear after a prompt change? Which ones remain stubborn?

In [ ]:

from collections import Counter, defaultdict

bucket_counts = Counter(record["bucket"] for record in annotated_run if record["bucket"] != "pass")
version_bucket_counts = defaultdict(Counter)
task_bucket_counts = defaultdict(Counter)

for record in annotated_run:
    if record["bucket"] == "pass":
        continue
    version_bucket_counts[record["version"]][record["bucket"]] += 1
    task_bucket_counts[record["task_type"]][record["bucket"]] += 1

print("Failure counts by bucket")
print_table([{"bucket": key, "count": value} for key, value in sorted(bucket_counts.items())])
print()

version_rows = []
for version, counter in sorted(version_bucket_counts.items()):
    for bucket, count in sorted(counter.items()):
        version_rows.append({"version": version, "bucket": bucket, "count": count})

print("Failure counts by version")
print_table(version_rows, headers=["version", "bucket", "count"])


## Severity buckets help prioritize the work

Two medium failures do not necessarily matter less than one critical failure, but they are not interchangeable. Severity lets you weight roadmap decisions by impact.

In [ ]:

severity_counts = Counter(record["severity"] for record in annotated_run if record["bucket"] != "pass")

weighted_burden = defaultdict(lambda: {"failures": 0, "severity_points": 0})
for record in annotated_run:
    if record["bucket"] == "pass":
        continue
    weighted_burden[record["version"]]["failures"] += 1
    weighted_burden[record["version"]]["severity_points"] += record["severity_score"]

severity_rows = [{"severity": key, "count": value} for key, value in sorted(severity_counts.items(), key=lambda item: severity_order[item[0]], reverse=True)]
burden_rows = [
    {
        "version": version,
        "failures": details["failures"],
        "severity_points": details["severity_points"],
        "average_severity": round(details["severity_points"] / details["failures"], 3),
    }
    for version, details in sorted(weighted_burden.items())
]

print("Severity distribution")
print_table(severity_rows, headers=["severity", "count"])
print()
print("Weighted burden by version")
print_table(burden_rows, headers=["version", "failures", "severity_points", "average_severity"])


## Summarize recurring patterns with simple signatures

A sophisticated system might cluster failures with embeddings or human review. For a lightweight notebook, simple signatures based on task type, bucket, and note keywords already reveal repetition.

In [ ]:

def normalize_text(text):
    characters = []
    for character in text.lower():
        characters.append(character if character.isalnum() else " ")
    return " ".join("".join(characters).split())

STOPWORDS = {
    "the", "a", "an", "and", "to", "for", "of", "in", "with", "on", "it",
    "is", "this", "that", "still", "into", "by", "or", "after"
}

def note_signature(text, max_terms=3):
    tokens = [token for token in normalize_text(text).split() if token not in STOPWORDS]
    return " ".join(tokens[:max_terms])

pattern_counter = Counter()
for record in annotated_run:
    if record["bucket"] == "pass":
        continue
    signature = f"{record['task_type']} | {record['bucket']} | {note_signature(record['notes'])}"
    pattern_counter[signature] += 1

pattern_rows = [{"pattern": pattern, "count": count} for pattern, count in pattern_counter.most_common()]
print_table(pattern_rows, headers=["pattern", "count"])


## Compare versions scenario by scenario

Version-level averages are useful, but paired scenario comparisons show exactly where a prompt helped, where it regressed, and where it stayed flat.

In [ ]:

records_by_scenario = defaultdict(dict)
for record in annotated_run:
    records_by_scenario[record["scenario"]][record["version"]] = record

comparison_rows = []
for scenario, versions in sorted(records_by_scenario.items()):
    baseline = versions["baseline"]
    prompt_v2 = versions["prompt_v2"]

    if baseline["bucket"] == "pass" and prompt_v2["bucket"] == "pass":
        change = "stable_pass"
    elif baseline["bucket"] != "pass" and prompt_v2["bucket"] == "pass":
        change = "improved"
    elif baseline["bucket"] == "pass" and prompt_v2["bucket"] != "pass":
        change = "regressed"
    elif baseline["bucket"] == prompt_v2["bucket"]:
        change = "same_failure"
    else:
        change = "different_failure"

    comparison_rows.append(
        {
            "scenario": scenario,
            "baseline_bucket": baseline["bucket"],
            "prompt_v2_bucket": prompt_v2["bucket"],
            "change": change,
        }
    )

print_table(comparison_rows, headers=["scenario", "baseline_bucket", "prompt_v2_bucket", "change"])


## Turn failure analysis into roadmap actions

The point of taxonomy work is not to create pretty labels. The point is to decide what to do next:

- prompt rewrite
- schema or formatting guardrail
- new regression tests
- policy rule
- additional human review

We can map buckets to likely interventions.

In [ ]:

ACTION_LIBRARY = {
    "wrong_label": {"recommended_action": "Add harder routing counterexamples and security-focused few-shot examples.", "owner": "prompt_design"},
    "missing_required_field": {"recommended_action": "Strengthen extraction instructions and add field-level regression tests.", "owner": "schema_and_eval"},
    "partial_answer": {"recommended_action": "Add checklist-style answer constraints and completeness tests.", "owner": "prompt_design"},
    "hallucinated_detail": {"recommended_action": "Add groundedness checks and unsupported-claim regression cases.", "owner": "eval_harness"},
    "formatting_error": {"recommended_action": "Tighten output contract and parser retry logic.", "owner": "interface_layer"},
    "policy_violation": {"recommended_action": "Add hard safety rules and an explicit account-verification policy gate.", "owner": "safety"},
    "other_failure": {"recommended_action": "Review manually and decide whether a new bucket is needed.", "owner": "analysis"},
}

bucket_summary = defaultdict(lambda: {"count": 0, "highest_severity": "low"})
for record in annotated_run:
    if record["bucket"] == "pass":
        continue
    bucket_summary[record["bucket"]]["count"] += 1
    current = bucket_summary[record["bucket"]]["highest_severity"]
    if severity_order[record["severity"]] > severity_order[current]:
        bucket_summary[record["bucket"]]["highest_severity"] = record["severity"]

roadmap_rows = []
for bucket, details in sorted(bucket_summary.items(), key=lambda item: (-severity_order[item[1]["highest_severity"]], -item[1]["count"], item[0])):
    roadmap_rows.append(
        {
            "bucket": bucket,
            "count": details["count"],
            "highest_severity": details["highest_severity"],
            "owner": ACTION_LIBRARY[bucket]["owner"],
            "recommended_action": ACTION_LIBRARY[bucket]["recommended_action"],
        }
    )

print_table(roadmap_rows, headers=["bucket", "count", "highest_severity", "owner", "recommended_action"])


## Produce a compact experiment summary

A useful report should fit on one screen: total pass rate, major failure families, version changes, and the next recommended interventions.

In [ ]:

total_pass_rate = sum(record["passed"] for record in annotated_run) / len(annotated_run)
pass_rate_by_version = defaultdict(lambda: {"passed": 0, "total": 0})

for record in annotated_run:
    pass_rate_by_version[record["version"]]["passed"] += record["passed"]
    pass_rate_by_version[record["version"]]["total"] += 1

summary_lines = [
    f"Overall pass rate: {round(total_pass_rate, 3)}",
]

for version, details in sorted(pass_rate_by_version.items()):
    rate = details["passed"] / details["total"]
    summary_lines.append(f"{version} pass rate: {round(rate, 3)}")

summary_lines.append("Top failure buckets:")
for row in roadmap_rows[:3]:
    summary_lines.append(f"- {row['bucket']} ({row['count']} cases, severity {row['highest_severity']})")

summary_text = "\n".join(summary_lines)
print(summary_text)

artifact_dir = Path("module_1_artifacts")
artifact_dir.mkdir(exist_ok=True)
summary_path = artifact_dir / "04_failure_analysis_summary.json"

with summary_path.open("w", encoding="utf-8") as handle:
    json.dump(
        {
            "overall_pass_rate": round(total_pass_rate, 3),
            "roadmap_rows": roadmap_rows,
            "scenario_comparison": comparison_rows,
        },
        handle,
        indent=2,
    )

print()
print("Saved:", summary_path)


## Takeaways

- error analysis gives meaning to benchmark scores
- a simple taxonomy already turns noisy failures into repeatable buckets
- severity buckets help prioritize what to fix first
- paired scenario review reveals improvements and regressions clearly
- the real goal is action: prompts, guardrails, tests, and policy rules that address the most important failures

With these four notebooks, Module 1 now covers why evals matter, how to build datasets, how to compute metrics, and how to turn failures into engineering decisions.